# 📊 Customer Segmentation using K-Means Clustering

## Project Overview
Customer segmentation is the process of dividing a customer base into distinct sub-groups of individuals that share similar demographic and behavioral characteristics (e.g., Age, Annual Income, Spending Score, Purchase Frequency, Recency, Monetary Value).

### Key Steps Covered in this Notebook:
1. **Exploratory Data Analysis (EDA)** and distributions.
2. **Data Preprocessing & Feature Scaling** (`StandardScaler` / `MinMaxScaler`).
3. **Determining Optimal K**: Within-Cluster Sum of Squares (WCSS / Elbow Method) & Silhouette Analysis.
4. **K-Means Model Fitting** and PCA 2D/3D Dimensionality Reduction.
5. **Cluster Profiling & Persona Generation** with targeted business marketing strategies.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Set plot style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset

In [ ]:
df = pd.read_csv('data/mall_customers.csv')
print("Dataset Shape:", df.shape)
df.head()

## 2. Exploratory Data Analysis & Feature Scaling

In [ ]:
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
X_scaled_df.head()

## 3. Elbow Method & Silhouette Score Analysis

In [ ]:
wcss = []
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

ax[0].plot(K_range, wcss, 'bo-', linewidth=2, markersize=8)
ax[0].set_title('Elbow Method (WCSS vs K)', fontsize=14)
ax[0].set_xlabel('Number of Clusters (K)')
ax[0].set_ylabel('WCSS')

ax[1].plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax[1].set_title('Silhouette Score vs K', fontsize=14)
ax[1].set_xlabel('Number of Clusters (K)')
ax[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

## 4. Train K-Means (K=5) & PCA 2D Cluster Visualization

In [ ]:
optimal_k = 5
kmeans_model = KMeans(n_clusters=optimal_k, init='k-means++', n_init=15, random_state=42)
df['Cluster'] = kmeans_model.fit_predict(X_scaled)

# PCA 2D Projection
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)
df['PCA1'] = pca_coords[:, 0]
df['PCA2'] = pca_coords[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='Cluster', palette='tab10', s=100, alpha=0.9)
plt.title('Customer Segments Visualized in PCA 2D Space', fontsize=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 5. Cluster Profiling & Business Insights

In [ ]:
cluster_profile = df.groupby('Cluster')[features].mean().round(2)
cluster_profile['Customer Count'] = df['Cluster'].value_counts()
cluster_profile['Percentage (%)'] = (cluster_profile['Customer Count'] / len(df) * 100).round(1)
cluster_profile